# ModPlant-LLM Notebook Interface

This notebook presents the `ModPlant-LLM` hybrid planning workflow as a notebook interface.

The notebook is organized into three workflow sections:

- `Settings`
- `Seed and Context`
- `Pipeline Execution`

This notebook runs the `Seed -> LLM -> FSA Checker -> FSA+BFS+OPT` workflow through reusable backend session logic.


## Section 1: Usage Notes

This notebook provides a notebook-native interaction style for the ModPlant-LLM workflow.

- Use the **Settings** section to pick the LoRA adapter, prewarm the model, and configure persistence and timeout behavior.
- Use **Seed and Context** to generate the ModPlant configuration, schedule, and reaction-rule data.
- Use **Pipeline Execution** to run the full `Seed -> LLM -> FSA Checker -> FSA+BFS+OPT` pipeline in a background thread.
- When the LLM result is feasible, the decision controls appear inside **Pipeline Execution** after **Check Result** and before **FSA+BFS+OPT Status**.

The default adapter path is `Model/3B-20260226_224018`. If you cloned a source-only copy without model weights, download the release ZIP that includes the adapter directories or provide a compatible PEFT adapter path.

The notebook updates widget outputs in place, so you can rerun the workflow inside Jupyter.


## Section 2: Setup And Shared State

Run the next cell once to resolve the project root, add the local package paths, and build the reusable widget state.


In [ ]:
from pathlib import Path
import os
import sys
from IPython.display import display

env_root = os.environ.get("MODPLANT_LLM_ROOT")
_candidate_roots = []
if env_root:
    _candidate_roots.append(Path(env_root).expanduser())
_candidate_roots.extend([
    Path.cwd(),
    Path.cwd() / "ModPlant-LLM",
])

PROJECT_ROOT = next((path.resolve() for path in _candidate_roots if (path / "ModPlant_ui_lib").is_dir()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not find ModPlant_ui_lib next to the notebook. Run Jupyter from the ModPlant-LLM folder or set MODPLANT_LLM_ROOT.")

for candidate in (PROJECT_ROOT, PROJECT_ROOT.parent):
    candidate_str = str(candidate)
    if candidate_str not in sys.path:
        sys.path.insert(0, candidate_str)

AUTO_PREWARM_DEFAULT_ADAPTER = True

from ModPlant_ui_lib.notebook_ui import ModPlantNotebookUI

ui = ModPlantNotebookUI(
    project_root=PROJECT_ROOT,
    auto_prewarm_default_adapter=AUTO_PREWARM_DEFAULT_ADAPTER,
).build()

PROJECT_ROOT


## Section 3: Settings

This section configures model and runtime settings.

Use it to:

- select or type a LoRA adapter directory,
- optionally open a browse dialog,
- prewarm the runtime,
- toggle temporary artifact persistence,
- adjust the single-run LLM timeout.


In [ ]:
display(ui.settings_section)

## Section 4: Seed And Context

This section provides generation controls and overview tables.

Use **Random Seed** if you want a new deterministic scenario, then **Generate From Seed** to populate:

- the ModPlant configuration table,
- the recipe schedule table,
- the reaction-rules table,
- the recipe summary line.


In [ ]:
display(ui.context_section)

## Section 5: Pipeline Execution

This section runs the LLM, checker, FSA+BFS+OPT, and runtime log workflow.

When you click **Start Calculation**, the notebook runs the pipeline in a background thread and updates:

- total / LLM / checker / FSA timing labels,
- LLM connect/process tables,
- FSA checker status and text output,
- the decision controls for feasible LLM results,
- FSA reference connect/process tables,
- runtime log text.


In [ ]:
display(ui.pipeline_section)

## Section 6: Re-running Safely

This notebook is designed to support repeated runs.

- You do not need to rebuild the widgets unless you restart the kernel.
- Each new pipeline run resets the notebook outputs before it starts.
- The backend thread is recreated per run, so you do not carry stale tables or stale logs into the next execution.
- If you change the adapter path, it is a good idea to run **Prewarm Model** again before the next pipeline run.

If you want a completely clean state, restart the kernel and rerun the setup cell.
